In [1]:
"""
Vignette OCR Inference Pipeline
"""

# ====================================================
# 1. SETUP & IMPORTS
# ====================================================
import sys, os, subprocess, glob, tarfile, shutil, warnings
warnings.filterwarnings("ignore")

def setup_offline():
    try:
        import ultralytics
        return
    except ImportError:
        archives = glob.glob('/kaggle/input/**/archive.tar.gz', recursive=True)
        if archives:
            tmp_dir = "/kaggle/working/pkg_tmp"
            os.makedirs(tmp_dir, exist_ok=True)
            with tarfile.open(archives[0], "r:gz") as tar:
                tar.extractall(path=tmp_dir)
            
            whl_dir = os.path.dirname(glob.glob(f"{tmp_dir}/**/*.whl", recursive=True)[0])
            
            subprocess.check_call([sys.executable, "-m", "pip", "install", "ultralytics", 
                                   "--no-index", f"--find-links={whl_dir}", 
                                   "--ignore-installed", "--no-deps"], 
                                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            shutil.rmtree(tmp_dir)
setup_offline()

import math
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import re
from PIL import Image
from torchvision import models, transforms
from tqdm import tqdm

try:
    from ultralytics import YOLO
    print("[System] Environment Ready. Libraries Loaded.")
except ImportError:
    print("[Error] Failed to initialize Ultralytics.")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
[System] Environment Ready. Libraries Loaded.


In [2]:
# ====================================================
# KAGGLE VERSION - Optimized for SVTR v3
# YOLO OBB -> Orientation -> Optimized Preprocessing -> OCR
# ====================================================

import os, math, re, warnings
warnings.filterwarnings("ignore")

import cv2
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms

from ultralytics import YOLO

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("🔧 Device:", DEVICE)

# ----------------------------------------------------
# PATHS (Kaggle)
# ----------------------------------------------------
DATA_DIR = Path("/kaggle/input/c/vignette-ocr")
TEST_DIR = DATA_DIR / "test"
SAMPLE_SUB = DATA_DIR / "sample_submission.csv"

YOLO_PATH = "/kaggle/input/yolo11-obb-vignette/pytorch/default/1/best.pt"
ORIENTATION_MODEL_PATH = "/kaggle/input/orientation/pytorch/default/1/orientation_classifier_best.pth"
OCR_MODEL_PATH = "/kaggle/input/svtr-ocr/pytorch/default/1/svtr_v3_unified_best.pth"

WORK_DIR = Path("/kaggle/working")
OUTPUT_SUBMISSION = WORK_DIR / "submission.csv"

if OUTPUT_SUBMISSION.exists():
    OUTPUT_SUBMISSION.unlink()
    print("🗑️ Deleted old submission.csv")

print("📁 TEST_DIR:", TEST_DIR)
print("📁 OUT:", OUTPUT_SUBMISSION)

# ----------------------------------------------------
# Load YOLO
# ----------------------------------------------------
model = YOLO(YOLO_PATH)
ENTETE = 0
MIN_CONF = 0.40
IMGSZ = 1024
PADDING = 5

image_paths = [p for p in TEST_DIR.glob("*.*") if p.suffix.lower() in [".jpg", ".jpeg", ".png"]]
print(f"📷 Found {len(image_paths)} images")

# ----------------------------------------------------
# SVTR v3 OCR model
# ----------------------------------------------------
IMG_HEIGHT, IMG_WIDTH = 64, 400
D_MODEL, NHEAD, NUM_LAYERS, DIM_FF, DROPOUT = 384, 8, 6, 1024, 0.15
ORI_IMG_SIZE = 224

CHAR_VECTOR = "-/0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"
idx_to_char = {i + 1: c for i, c in enumerate(CHAR_VECTOR)}
NUM_CLASSES = len(CHAR_VECTOR) + 1

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000) / d_model))
        pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * div), torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(1))

    def forward(self, x):
        return self.dropout(x + self.pe[: x.size(0)])

class SVTR_v3(nn.Module):
    def __init__(self, vocab_size, d_model=384, nhead=8, num_layers=6, dim_ff=1024, dropout=0.15):
        super().__init__()
        resnet = models.resnet34(weights=None)
        for layer in [resnet.layer2, resnet.layer3, resnet.layer4]:
            layer[0].conv1.stride = (2, 1)
            layer[0].downsample[0].stride = (2, 1)
        self.backbone = nn.Sequential(
            resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool,
            resnet.layer1, resnet.layer2, resnet.layer3, resnet.layer4
        )
        self.pool = nn.AdaptiveAvgPool2d((1, None))
        self.proj = nn.Conv2d(512, d_model, 1)
        self.pos = PositionalEncoding(d_model, dropout=dropout)
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model, nhead, dim_ff, dropout, batch_first=False),
            num_layers
        )
        self.classifier = nn.Linear(d_model, vocab_size)

    def forward(self, x):
        x = self.proj(self.pool(self.backbone(x))).squeeze(2).permute(2, 0, 1)
        x = self.pos(x)
        x = self.transformer(x)
        x = self.classifier(x)
        return F.log_softmax(x, dim=2)

print("Loading OCR model...")
ocr_model = SVTR_v3(NUM_CLASSES, D_MODEL, NHEAD, NUM_LAYERS, DIM_FF, DROPOUT).to(DEVICE)

ckpt = torch.load(OCR_MODEL_PATH, map_location=DEVICE)
if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
    state_dict = ckpt["model_state_dict"]
    if "vocab" in ckpt:
        _, idx_to_char_ckpt, vocab_size = ckpt["vocab"]
        idx_to_char = idx_to_char_ckpt
        print(f"  ✓ Loaded vocab: {vocab_size} classes")
else:
    state_dict = ckpt

ocr_model.load_state_dict(state_dict, strict=False)
ocr_model.eval()
print("✅ OCR loaded")

# ----------------------------------------------------
# Orientation model
# ----------------------------------------------------
class OrientationClassifier:
    def __init__(self, model_path, device):
        self.device = device
        from torchvision.models import resnet18
        checkpoint = torch.load(model_path, map_location=device)
        state_dict = checkpoint["model_state_dict"] if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint else checkpoint

        base_model = resnet18(weights=None)
        in_features = base_model.fc.in_features

        if "fc.1.weight" in state_dict and "fc.4.weight" in state_dict:
            hidden_dim = state_dict["fc.1.weight"].shape[0]
            fc_layers = nn.ModuleDict({
                "1": nn.Linear(in_features, hidden_dim),
                "2": nn.ReLU(),
                "3": nn.Dropout(0.5),
                "4": nn.Linear(hidden_dim, 2),
            })
            fc_layers["1"].weight.data = state_dict["fc.1.weight"]
            fc_layers["1"].bias.data = state_dict["fc.1.bias"]
            fc_layers["4"].weight.data = state_dict["fc.4.weight"]
            fc_layers["4"].bias.data = state_dict["fc.4.bias"]

            class CustomFC(nn.Module):
                def __init__(self, layers):
                    super().__init__()
                    self.fc = layers
                def forward(self, x):
                    for k in ["1", "2", "3", "4"]:
                        x = self.fc[k](x)
                    return x

            base_model.fc = CustomFC(fc_layers)
            resnet_state = {k: v for k, v in state_dict.items() if not k.startswith("fc.")}
            base_model.load_state_dict(resnet_state, strict=False)
        else:
            base_model.fc = nn.Linear(in_features, 2)
            base_model.load_state_dict(state_dict, strict=False)

        self.model = base_model.to(device).eval()
        self.transform = transforms.Compose([
            transforms.Resize((ORI_IMG_SIZE, ORI_IMG_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
        ])

    def predict(self, pil_image):
        x = self.transform(pil_image).unsqueeze(0).to(self.device)
        with torch.no_grad():
            out = self.model(x)
            pred = out.argmax(1).item()
        return pred == 1

print("Loading orientation model...")
orientation_model = OrientationClassifier(ORIENTATION_MODEL_PATH, DEVICE)
print("✅ Orientation loaded")

# ----------------------------------------------------
# Helpers - Optimized for SVTR v3 training
# ----------------------------------------------------
def make_horizontal(image: Image.Image):
    w, h = image.size
    return image.rotate(90, expand=True) if h > w else image

def enhance_image(img: Image.Image):
    """Optimized preprocessing for SVTR v3 (based on training analysis)"""
    img_np = np.array(img)
    
    # Upscale small images
    h, w = img_np.shape[:2]
    if w < 400 or h < 100:
        img_np = cv2.resize(img_np, (w * 2, h * 2), interpolation=cv2.INTER_LANCZOS4)
    
    # 1. Light Denoise (h=3, model trained on clean images)
    img_np = cv2.fastNlMeansDenoisingColored(img_np, None, 3, 3, 7, 15)
    
    # 2. CLAHE (clipLimit=2.5, matches training contrast_limit=0.2)
    lab = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    lab[:,:,0] = clahe.apply(lab[:,:,0])
    img_np = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
    
    # 3. Light Sharpen (don't destroy characters)
    kernel = np.array([[0, -0.5, 0],
                       [-0.5, 3, -0.5],
                       [0, -0.5, 0]])
    img_np = cv2.filter2D(img_np, -1, kernel)
    
    return Image.fromarray(np.clip(img_np, 0, 255).astype(np.uint8))

def preprocess_for_ocr(image: Image.Image):
    img = np.array(image)
    if len(img.shape) == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    elif img.shape[2] == 4:
        img = cv2.cvtColor(img, cv2.COLOR_RGBA2RGB)
    img = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT))
    img = (img.astype(np.float32) / 255.0 - np.array([0.485,0.456,0.406])) / np.array([0.229,0.224,0.225])
    return torch.from_numpy(img).permute(2,0,1).float().unsqueeze(0).to(DEVICE)

def decode_ctc(preds):
    _, idx = preds.max(2)
    chars, prev = [], -1
    for j in idx[:, 0].detach().cpu().numpy():
        if j != 0 and j != prev and j in idx_to_char:
            chars.append(idx_to_char[j])
        prev = j
    return "".join(chars)

def clean_text(text):
    return re.sub(r"[^A-Z0-9/\-]", "", text.upper())

# ----------------------------------------------------
# MAIN LOOP
# ----------------------------------------------------
results = []
stats = {"rotated": 0, "not_rotated": 0}

pbar = tqdm(image_paths, desc="YOLO → OCR", unit="img")

for img_path in pbar:
    image = cv2.imread(str(img_path))
    if image is None:
        results.append({"id": img_path.name, "text": ""})
        continue

    final_crop = None

    try:
        preds = model.predict(source=image, task="obb", imgsz=IMGSZ, conf=0.25, classes=[ENTETE], save=False, verbose=False)
        if preds and preds[0].obb is not None and len(preds[0].obb) > 0:
            r = preds[0]
            confs = r.obb.conf.cpu().numpy()
            top_idx = int(np.argmax(confs))
            if float(confs[top_idx]) >= MIN_CONF:
                x, y, w, h, angle = map(float, r.obb.xywhr[top_idx].tolist())
                angle_deg = angle * 180.0 / math.pi
                M = cv2.getRotationMatrix2D((x, y), angle_deg, 1.0)
                rotated = cv2.warpAffine(image, M, (image.shape[1], image.shape[0]))
                x1, y1 = max(0, int(x - w/2) - PADDING), max(0, int(y - h/2) - PADDING)
                x2, y2 = min(rotated.shape[1], int(x + w/2) + PADDING), min(rotated.shape[0], int(y + h/2) + PADDING)
                crop = rotated[y1:y2, x1:x2]
                if crop.size > 0:
                    final_crop = crop
    except:
        pass

    if final_crop is None:
        final_crop = image

    pil_img = Image.fromarray(cv2.cvtColor(final_crop, cv2.COLOR_BGR2RGB))
    pil_img = make_horizontal(pil_img)

    if orientation_model.predict(pil_img):
        pil_img = pil_img.rotate(180)
        stats["rotated"] += 1
    else:
        stats["not_rotated"] += 1

    pil_img = enhance_image(pil_img)

    with torch.no_grad():
        out = ocr_model(preprocess_for_ocr(pil_img))

    text = clean_text(decode_ctc(out))
    results.append({"id": img_path.name, "text": text})

pbar.close()
print("📊 Rotation stats:", stats)

# ----------------------------------------------------
# Build submission
# ----------------------------------------------------
df = pd.DataFrame(results, columns=["id", "text"])

def clean_id(value):
    value = str(value).strip()
    value = re.sub(r"_entete.*$", "", value, flags=re.IGNORECASE)
    if not value.lower().endswith(".jpg"):
        value += ".jpg"
    return value

if len(df) > 0:
    df["id"] = df["id"].apply(clean_id)

sample = pd.read_csv(SAMPLE_SUB)
id_col, text_col = sample.columns[0], sample.columns[1]

merged = sample[[id_col]].merge(df, left_on=id_col, right_on="id", how="left")
merged[text_col] = merged["text"].fillna("")
submission = merged[[id_col, text_col]]

submission.to_csv(OUTPUT_SUBMISSION, index=False)
print("✅ Saved:", OUTPUT_SUBMISSION)
submission.head(10)

🔧 Device: cuda
📁 TEST_DIR: /kaggle/input/c/vignette-ocr/test
📁 OUT: /kaggle/working/submission.csv
📷 Found 674 images
Loading OCR model...
  ✓ Loaded vocab: 39 classes
✅ OCR loaded
Loading orientation model...
✅ Orientation loaded


YOLO → OCR: 100%|██████████| 674/674 [01:41<00:00,  6.66img/s]

📊 Rotation stats: {'rotated': 356, 'not_rotated': 318}
✅ Saved: /kaggle/working/submission.csv


,id,text
0,0016f1ad9e7f16aa6dd2b8c9c13cb922.jpg,LOTK461FAB03/2019EXP03/2021109/17025/092
1,005e28f75a93bc4de0e9e11373bdca47.jpg,LOT395FABFEV20EXPFEV23
2,007f7b00eb0a66cabcd72007358e3660.jpg,LOT744550FAB07/2020EXP0/2007/202DE11H12/023
3,009a43c2870949af417f769b90ee20cf.jpg,LOTX3A37/11FAB09/2018EXP09/2021DE14/98/11H00/076
4,014d3d8a2ef0d4deb88a83357781745c.jpg,LOTH5154FABOCT/2019PERSEP/2021DE99/17B109/096
5,0174f52895414f9f2b5842547803f533.jpg,LOTN4438/1FF1FABJAN2020EXPJAN2022
6,02a45854f6bdaff85587d8017a344ff9.jpg,LOT077FAB02/2017EXP12/2020
7,02d5c041d5aebd133a34b7cb5e679557.jpg,LOTX34549FAB07/2018EXP06/2021DEE98/11H028/030/C
8,02ff1c092c2f1d483916e898ef740392.jpg,LOT359246FAB05-2019PER04-2024
9,0396159a179dc766efd00a86fa040f02.jpg,LOTC114FAB03/2019PER02/2022DE07/00/03B042/392
